# Adaptive Block-Based Percentile Thresholding with Capacity-Aware RGB Allocation for Reversible Data Hiding

**Blok:** 3×4 = 12 pixel | **Ref positions (1-indexed):** 3, 5, 10 → **(0-indexed):** 2, 4, 9  
**Payload:** 1-100 **Kilobit** 
**Threshold Mode:** Adaptive block

---

- **Red (32%), Green (33%), Blue (35%)**
- Tujuan: Maksimalkan kapasitas di channel least-sensitive, jaga quality
- Weight didapat dari estimasi capacity check

### Adaptive P10-90 Percentile
- **Range:** P10 ≤ |diff| ≤ P90 (middle 80% dari distribusi |diff|)
- **Keuntungan:** Adaptif konten, menghindari pixel ekstrem
- **Trade-off:** Kapasitas lebih tinggi, tapi perubahan pixel lebih besar → PSNR bisa turun
- **Solusi:** Optimasi capacity-aware allocation untuk balance


In [1]:
# ============================================================
# CELL 1: Import
# ============================================================
import numpy as np
import cv2
import os
import json
import csv
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim_fn
import pandas as pd

print('Libraries loaded.')

Libraries loaded.


In [3]:
# ============================================================
# CELL 2: CONFIG — BLOCK P10–90 (FINAL CLEAN)
# ============================================================
import os
import numpy as np

DATASET_PATH = r'D:\KBJ-Komputasi Jaringan\data hiding paper\propose fix\dataset'
OUTPUT_PATH  = r'D:\KBJ-Komputasi Jaringan\data hiding paper\propose fix\output_p1090'
PAYLOAD_PATH = r'D:\KBJ-Komputasi Jaringan\data hiding paper\propose fix\payload'
# ============================================================
# VALIDATION
# ============================================================
if not os.path.isdir(DATASET_PATH):
    raise FileNotFoundError(
        f'Dataset folder tidak ditemukan:\n{DATASET_PATH}'
    )

if not os.path.isdir(PAYLOAD_PATH):
    raise FileNotFoundError(
        f'Payload folder tidak ditemukan:\n{PAYLOAD_PATH}'
    )

os.makedirs(OUTPUT_PATH, exist_ok=True)
# ============================================================
# IMAGE LIST
# ============================================================
SUPPORTED_EXTENSIONS = (
    '.png',
    '.jpg',
    '.jpeg',
    '.bmp',
    '.tif',
    '.tiff'
)

IMAGE_FILES = sorted([
    f for f in os.listdir(DATASET_PATH)
    if f.lower().endswith(SUPPORTED_EXTENSIONS)
])

if len(IMAGE_FILES) == 0:
    raise FileNotFoundError(
        f'Tidak ada image pada folder:\n{DATASET_PATH}'
    )
# ============================================================
# DATASET
# ============================================================
print(f'Jumlah image : {len(IMAGE_FILES)}')

for f in IMAGE_FILES[:10]:
    print('  -', f)

if len(IMAGE_FILES) > 10:
    print(f'  ... +{len(IMAGE_FILES)-10} image lainnya')
# ============================================================
# BLOK KONFIGURASI (3×4)
# ============================================================
# Layout:
#   [  0    1   [2]   3 ]
#   [ [4]   5    6    7 ]
#   [  8   [9]  10   11 ]
#
# REF (tidak diubah):
#   pos 2, 4, 9 (0-indexed)
#
# Tujuan:
#   - menjaga reversibility
#   - menghindari pola simetris (anti steganalisis)
# ============================================================

BLOCK_ROWS = 3
BLOCK_COLS = 4
BLOCK_SIZE = BLOCK_ROWS * BLOCK_COLS   # = 12

REF_INDICES = {2, 4, 9}

REF_RULES = [
    (range(0, 2),   2),    # pos 0,1   → ref=2
    (range(3, 4),   4),    # pos 3     → ref=4
    (range(5, 9),   9),    # pos 5–8   → ref=9
    (range(10, 12), 9),    # pos 10–11 → ref=9
]

# Posisi yang boleh di-embed (selain ref)
EMBED_POSITIONS = [i for i in range(BLOCK_SIZE) if i not in REF_INDICES]

CHANNEL_WEIGHT = {'R': 0.32,'G': 0.33,'B': 0.35}

# ============================================================
# THRESHOLD MODE — BLOCK ADAPTIVE P10–90
# ============================================================
# Untuk SETIAP BLOCK:
#   1. Hitung semua |diff|
#   2. Ambil:
#        P10 = percentile 10%
#        P90 = percentile 90%
#   3. Embed hanya jika:
#        P10 ≤ |diff| ≤ P90
#
# Artinya:
#   - Hanya middle 80% distribusi yang dipakai
#   - Menghindari:
#       • diff terlalu kecil (tidak stabil)
#       • diff terlalu besar (distorsi tinggi)
#
# Keunggulan:
#   ✔ adaptif terhadap tekstur lokal
#   ✔ lebih robust dari threshold global
# ============================================================
THRESHOLD_MODE = 'block_p10_90'

# Fallback (tidak dipakai di mode ini, tapi disiapkan)
# T_MAX = {'G': 3, 'R': 3, 'B': 3}

# ============================================================
# SAFE DE BOUND
# Setelah DE:
#    h' = 2h + b
# Embed hanya jika:
#    |h'| <= MAX_H
#
# Ini threshold distortion control utama
# ============================================================
MAX_H = {'R': 10,'G': 12,'B': 12}
# ============================================================
# PAYLOAD (dalam KILOBIT)
# 1 Kbit = 1024 bit
# ============================================================
PAYLOAD_SIZES_KBIT = [1, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
# ============================================================
# PRINT INFO
# ============================================================

print(f'Dataset : {DATASET_PATH}')
print(f'Output  : {OUTPUT_PATH}')
print()

print(f'Blok    : {BLOCK_ROWS}x{BLOCK_COLS} = {BLOCK_SIZE} piksel')
print(f'Ref idx : {sorted(REF_INDICES)} (0-indexed)')
print(f'Embed   : {len(EMBED_POSITIONS)} piksel per block')
print()

print(f'Channel Weight : G={CHANNEL_WEIGHT["G"]*100:.0f}% '
      f'R={CHANNEL_WEIGHT["R"]*100:.0f}% '
      f'B={CHANNEL_WEIGHT["B"]*100:.0f}%')
print()

print(f'Mode    : {THRESHOLD_MODE}')
print('P10-90  : Dihitung PER BLOCK (bukan global)')
print()

print(f'Payload : {PAYLOAD_SIZES_KBIT} Kbit')
print()

# ============================================================
# ESTIMASI KAPASITAS (UPPER BOUND)
# ============================================================
print('\n=== ESTIMASI KAPASITAS DATASET ===')

dataset_capacity = []

for img_name in IMAGE_FILES:

    img = cv2.imread(
        os.path.join(DATASET_PATH, img_name),
        cv2.IMREAD_COLOR
    )

    if img is None:
        continue

    H, W = img.shape[:2]

    total_px = H * W
    n_blocks = total_px // BLOCK_SIZE

    max_bits_per_ch = n_blocks * len(EMBED_POSITIONS)
    max_kbit_total = max_bits_per_ch * 3 / 1024

    dataset_capacity.append(max_kbit_total)

    print(
        f'{img_name:<25} '
        f'{W:4d}x{H:<4d} '
        f'≈ {max_kbit_total:8.1f} Kbit'
    )

print('\n=== RINGKASAN DATASET ===')
print(f'Jumlah image : {len(dataset_capacity)}')
print(f'Min capacity : {min(dataset_capacity):.1f} Kbit')
print(f'Max capacity : {max(dataset_capacity):.1f} Kbit')
print(f'Avg capacity : {np.mean(dataset_capacity):.1f} Kbit')

Jumlah image : 11
  - Abdomal.jpg
  - Chest.jpg
  - Hand.jpg
  - Head.jpg
  - Leg.jpg
  - airplane.tiff
  - baboon.tiff
  - house.tiff
  - pepper.tiff
  - sailboat.tiff
  ... +1 image lainnya
Dataset : D:\KBJ-Komputasi Jaringan\data hiding paper\propose fix\dataset
Output  : D:\KBJ-Komputasi Jaringan\data hiding paper\propose fix\output_p1090

Blok    : 3x4 = 12 piksel
Ref idx : [2, 4, 9] (0-indexed)
Embed   : 9 piksel per block

Channel Weight : G=33% R=32% B=35%

Mode    : block_p10_90
P10-90  : Dihitung PER BLOCK (bukan global)

Payload : [1, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100] Kbit


=== ESTIMASI KAPASITAS DATASET ===
Abdomal.jpg                512x512  ≈    576.0 Kbit
Chest.jpg                  512x512  ≈    576.0 Kbit
Hand.jpg                   512x512  ≈    576.0 Kbit
Head.jpg                   512x512  ≈    576.0 Kbit
Leg.jpg                    512x512  ≈    576.0 Kbit
airplane.tiff              512x512  ≈    576.0 Kbit
baboon.tiff                512x512  ≈    576.0 Kbit
h

In [4]:
# ============================================================
# CELL 3: Helper Functions (CLEAN VERSION)
# ============================================================

def get_ref_idx(pos: int) -> int:
    for positions, ref_idx in REF_RULES:
        if pos in positions:
            return ref_idx
    raise ValueError(f'Posisi {pos} tidak valid atau adalah referensi')


EMBED_POSITIONS = [p for p in range(BLOCK_SIZE) if p not in REF_INDICES]

print(f'Posisi embedding per blok : {EMBED_POSITIONS} ({len(EMBED_POSITIONS)} posisi)')

Posisi embedding per blok : [0, 1, 3, 5, 6, 7, 8, 10, 11] (9 posisi)


In [5]:
fname = 'random-binary_1Kb.txt'

with open(
    os.path.join(PAYLOAD_PATH, fname),
    'r'
) as f:

    content = f.read()

print(content[:500])

1	1	0	1	1	0	0	1	1	1	0	1	1	0	1	0	0	1	1	1	1	0	1	1	1	1	1	0	1	0	1	0	0	0	0	1	1	0	1	0	0	0	1	1	0	0	0	1	1	1	0	1	1	0	0	0	1	0	1	0	1	0	1	1	1	1	1	0	0	0	1	0	1	0	1	0	0	0	1	0	0	1	1	1	1	0	1	1	0	1	0	0	1	1	1	0	1	0	0	0	0	1	0	1	0	1	0	1	1	1	0	0	0	1	0	1	1	1	0	0	0	1	0	1	1	1	0	0	0	1	0	1	0	0	0	0	1	1	1	0	1	1	0	1	0	0	0	0	0	0	0	0	1	1	0	0	0	1	0	0	1	0	0	0	0	0	1	1	1	0	0	0	1	0	0	0	1	1	1	0	1	0	1	0	1	0	0	1	1	0	1	1	0	0	0	0	1	1	1	1	1	0	1	1	0	1	1	1	1	1	0	0	0	0	1	0	0	0	0	0	0	1	0	0	1	1	0	0	0	0	1	0	1	1	0	0	0	0	0	1	0	0	1	0	1	1	0	1	0	0	


In [6]:
fname = os.path.join(
    PAYLOAD_PATH,
    'random-binary_10Kb.txt'
)

with open(fname,'r') as f:
    content = f.read()

print(len(content.split()))

10000


In [7]:
# ============================================================
# CELL 4: Konversi Bit
# ============================================================

def text_to_bits(text: str) -> list:
    """String UTF-8 → list bit int (0/1)."""
    bits = []
    for byte in text.encode('utf-8'):
        for i in range(7, -1, -1):
            bits.append((byte >> i) & 1)
    return bits


def bits_to_text(bits: list) -> str:
    """List bit → string (8 bit per byte)."""
    chars = []
    for i in range(0, len(bits) - 7, 8):
        byte = 0
        for j in range(8):
            byte = (byte << 1) | int(bits[i + j])
        chars.append(chr(byte))
    return ''.join(chars)


def generate_bits_for_payload(payload_kbit):

    n_bits = payload_kbit * 1000

    fname = f'random-binary_{payload_kbit}Kb.txt'
    fpath = os.path.join(PAYLOAD_PATH, fname)

    if not os.path.exists(fpath):
        raise FileNotFoundError(
            f'Payload tidak ditemukan:\n{fpath}'
        )

    with open(fpath, 'r') as f:
        content = f.read()

    tokens = content.split()

    bits = [int(x) for x in tokens]

    if len(bits) < n_bits:

        raise ValueError(
            f'{fname} hanya memiliki '
            f'{len(bits)} bit '
            f'(butuh {n_bits})'
        )

    bits = bits[:n_bits]

    print(
        f'Payload loaded: '
        f'{fname} ({len(bits)} bit)'
    )

    return bits

# Uji konversi teks
_t = 'Hello v5 P1090!'
assert bits_to_text(text_to_bits(_t)) == _t
print(f'Konversi teks OK: "{_t}"')

# Uji payload
_b = generate_bits_for_payload(10)

print(
    f'Payload 10 Kbit: '
    f'{len(_b)} bit, '
    f'sample={_b[:10]}'
)

Konversi teks OK: "Hello v5 P1090!"
Payload loaded: random-binary_10Kb.txt (10000 bit)
Payload 10 Kbit: 10000 bit, sample=[1, 0, 1, 0, 1, 1, 0, 0, 0, 0]


In [8]:
# ============================================================
# CELL 5: embed_channel
# BLOCK P10–90 + SAFE DE
# ============================================================

def embed_channel(channel: np.ndarray,
                  bits: list,
                  ch_name: str):

    H, W = channel.shape

    vec = channel.flatten().astype(np.int32).tolist()

    rm = [0] * len(vec)

    total_px = H * W
    n_blocks = total_px // BLOCK_SIZE

    bit_idx = 0
    n_bits = 0

    MAX_H_CH = MAX_H[ch_name]

    # optional metadata
    block_p10 = []
    block_p90 = []

    for b in range(n_blocks):

        start = b * BLOCK_SIZE
        end = start + BLOCK_SIZE

        block = vec[start:end]

        if len(block) < BLOCK_SIZE:
            continue

        # ----------------------------------------
        # hitung distribusi |diff|
        # ----------------------------------------
        diffs = []

        for pos in EMBED_POSITIONS:

            ref_i = get_ref_idx(pos)

            ref = block[ref_i]
            cur = block[pos]

            diff = cur - ref

            diffs.append(abs(diff))

        if len(diffs) == 0:
            continue

        p10 = float(np.percentile(diffs, 10))
        p90 = float(np.percentile(diffs, 90))

        block_p10.append(p10)
        block_p90.append(p90)

        # ----------------------------------------
        # embedding
        # ----------------------------------------
        for pos in EMBED_POSITIONS:

            if bit_idx >= len(bits):
                break

            ref_i = get_ref_idx(pos)

            ref = block[ref_i]
            cur = block[pos]

            diff = cur - ref
            adiff = abs(diff)

            # ----------------------------
            # P10-P90 filter
            # ----------------------------
            if not (p10 <= adiff <= p90):
                continue

            bit = int(bits[bit_idx])

            # ----------------------------
            # DE expansion
            # ----------------------------
            new_diff = diff * 2 + bit

            if abs(new_diff) > MAX_H_CH:
                continue

            stego = ref + new_diff

            if stego < 0 or stego > 255:
                continue

            # ----------------------------
            # apply
            # ----------------------------
            block[pos] = stego

            vec[start + pos] = stego

            rm[start + pos] = 1

            bit_idx += 1
            n_bits += 1

        if bit_idx >= len(bits):
            break

    stego_ch = np.array(
        vec,
        dtype=np.uint8
    ).reshape(H, W)

    rm_map = np.array(
        rm,
        dtype=np.uint8
    ).reshape(H, W)
    
    block_meta = {
    'channel': ch_name,
    'embedded_bits': n_bits,
    'requested_bits': len(bits),
    'utilization': (
        n_bits / len(bits)
        if len(bits) > 0 else 0
    ),
    'num_blocks': n_blocks,
    'p10': block_p10,
    'p90': block_p90
    }

    print(
        f'[{ch_name}] '
        f'Embedded {n_bits}/{len(bits)} bits  '
        f'({100*n_bits/max(len(bits),1):.2f}%) '
    )

    return (stego_ch, rm_map, n_bits, block_meta)

In [9]:
# ============================================================
# CELL 5A: Capacity Check
# BLOCK P10–90 + SAFE DE
# ============================================================

def check_capacity(channel: np.ndarray,
                   ch_name: str):

    if ch_name not in MAX_H:
        raise ValueError(
            f'Unknown channel: {ch_name}'
        )

    H, W = channel.shape

    vec = channel.flatten().astype(np.int32).tolist()

    total_px = H * W
    n_blocks = total_px // BLOCK_SIZE

    total_candidates = 0
    threshold_skips = 0
    maxh_skips = 0
    overflow_skips = 0
    effective_capacity = 0

    MAX_H_CH = MAX_H[ch_name]

    for b in range(n_blocks):

        start = b * BLOCK_SIZE
        end = start + BLOCK_SIZE

        block = vec[start:end]

        if len(block) < BLOCK_SIZE:
            continue

        # ------------------------------------
        # hitung distribusi |diff|
        # ------------------------------------
        diffs = []

        for pos in EMBED_POSITIONS:

            ref_i = get_ref_idx(pos)

            ref = block[ref_i]
            cur = block[pos]

            diffs.append(abs(cur - ref))

        if len(diffs) == 0:
            continue

        p10 = float(np.percentile(diffs, 10))
        p90 = float(np.percentile(diffs, 90))

        # ------------------------------------
        # simulasi kapasitas
        # ------------------------------------
        for pos in EMBED_POSITIONS:

            total_candidates += 1

            ref_i = get_ref_idx(pos)

            ref = block[ref_i]
            cur = block[pos]

            diff = cur - ref
            adiff = abs(diff)

            # P10-P90 filter
            if not (p10 <= adiff <= p90):
                threshold_skips += 1
                continue

            # --------------------------------
            # cek kedua kemungkinan bit
            # bit=0 dan bit=1
            # --------------------------------
            embeddable = False

            for bit in (0, 1):

                new_diff = diff * 2 + bit

                if abs(new_diff) > MAX_H_CH:
                    continue

                stego = ref + new_diff

                if stego < 0 or stego > 255:
                    continue

                embeddable = True
                break

            if embeddable:
                effective_capacity += 1
            else:
                # klasifikasi alasan gagal
                if abs(diff * 2 + 1) > MAX_H_CH:
                    maxh_skips += 1
                else:
                    overflow_skips += 1

    utilization = (
        effective_capacity / total_candidates
        if total_candidates > 0
        else 0
    )

    report = {
        'channel': ch_name,
        'total_candidates': total_candidates,
        'threshold_skips': threshold_skips,
        'maxh_skips': maxh_skips,
        'overflow_skips': overflow_skips,
        'effective_capacity': effective_capacity,
        'utilization': utilization
    }

    return report

In [10]:
# ============================================================
# CELL 5B: Capacity Analysis Report
# ============================================================

import pandas as pd
import cv2

def check_image_capacity(rgb_img):

    reports = {}

    reports['R'] = check_capacity(
        rgb_img[:, :, 0],
        'R'
    )

    reports['G'] = check_capacity(
        rgb_img[:, :, 1],
        'G'
    )

    reports['B'] = check_capacity(
        rgb_img[:, :, 2],
        'B'
    )

    total_capacity = (
        reports['R']['effective_capacity']
        + reports['G']['effective_capacity']
        + reports['B']['effective_capacity']
    )

    return reports, total_capacity


# ============================================================
# ANALISIS SEMUA IMAGE
# ============================================================

for img_name in IMAGE_FILES:

    img_path = os.path.join(DATASET_PATH, img_name)

    img_bgr = cv2.imread(
        img_path,
        cv2.IMREAD_COLOR
    )

    img_rgb = cv2.cvtColor(
        img_bgr,
        cv2.COLOR_BGR2RGB
    )

    reports, total_capacity = check_image_capacity(
        img_rgb
    )

    print('=' * 70)
    print(f'Loaded image: {img_name}')
    print(f'Shape: {img_rgb.shape}')

    # --------------------------------------------------------
    # ratio aktual
    # --------------------------------------------------------
    cap_r = reports['R']['effective_capacity']
    cap_g = reports['G']['effective_capacity']
    cap_b = reports['B']['effective_capacity']

    ratio_r = cap_r / total_capacity
    ratio_g = cap_g / total_capacity
    ratio_b = cap_b / total_capacity

    print('\n=== CAPACITY vs CHANNEL TARGET ===')

    print(
        f'G: actual={ratio_g:.4f} | '
        f'target={CHANNEL_WEIGHT["G"]:.4f} | '
        f'diff={ratio_g-CHANNEL_WEIGHT["G"]:+.4f}'
    )

    print(
        f'R: actual={ratio_r:.4f} | '
        f'target={CHANNEL_WEIGHT["R"]:.4f} | '
        f'diff={ratio_r-CHANNEL_WEIGHT["R"]:+.4f}'
    )

    print(
        f'B: actual={ratio_b:.4f} | '
        f'target={CHANNEL_WEIGHT["B"]:.4f} | '
        f'diff={ratio_b-CHANNEL_WEIGHT["B"]:+.4f}'
    )

    # --------------------------------------------------------
    # dataframe
    # --------------------------------------------------------
    rows = []

    for ch in ['G', 'R', 'B']:

        rep = reports[ch]

        actual_ratio = (
            rep['effective_capacity']
            / total_capacity
        )

        rows.append({
            'Channel': ch,
            'Total candidates': rep['total_candidates'],
            'Threshold skips': rep['threshold_skips'],
            'MAX_H skips': rep['maxh_skips'],
            'Overflow skips': rep['overflow_skips'],
            'Effective capacity': rep['effective_capacity'],
            'Effective %':
                round(
                    100 * rep['effective_capacity']
                    / rep['total_candidates'],
                    2
                ),
            'Actual Ratio':
                round(actual_ratio, 4),
            'Channel Target':
                CHANNEL_WEIGHT[ch]
        })

    rows.append({
        'Channel': 'TOTAL',
        'Total candidates':
            sum(reports[c]['total_candidates']
                for c in ['G','R','B']),
        'Threshold skips':
            sum(reports[c]['threshold_skips']
                for c in ['G','R','B']),
        'MAX_H skips':
            sum(reports[c]['maxh_skips']
                for c in ['G','R','B']),
        'Overflow skips':
            sum(reports[c]['overflow_skips']
                for c in ['G','R','B']),
        'Effective capacity':
            total_capacity,
        'Effective %':
            round(
                100 * total_capacity /
                sum(
                    reports[c]['total_candidates']
                    for c in ['G','R','B']
                ),
                2
            ),
        'Actual Ratio': 1.0,
        'Channel Target': '-'
    })

    df = pd.DataFrame(rows)

    display(df)

    print(
        f'Total Capacity = '
        f'{total_capacity:,} bit '
        f'({total_capacity/1024:.2f} Kbit)'
    )

    print()

Loaded image: Abdomal.jpg
Shape: (512, 512, 3)

=== CAPACITY vs CHANNEL TARGET ===
G: actual=0.3378 | target=0.3300 | diff=+0.0078
R: actual=0.3244 | target=0.3200 | diff=+0.0044
B: actual=0.3378 | target=0.3500 | diff=-0.0122


,Channel,Total candidates,Threshold skips,MAX_H skips,Overflow skips,Effective capacity,Effective %,Actual Ratio,Channel Target
0,G,196605,19938,22648,1368,152651,77.64,0.3378,0.33
1,R,196605,19938,28812,1238,146617,74.57,0.3244,0.32
2,B,196605,19938,22648,1368,152651,77.64,0.3378,0.35
3,TOTAL,589815,59814,74108,3974,451919,76.62,1.0000,-


Total Capacity = 451,919 bit (441.33 Kbit)

Loaded image: Chest.jpg
Shape: (512, 512, 3)

=== CAPACITY vs CHANNEL TARGET ===
G: actual=0.3361 | target=0.3300 | diff=+0.0061
R: actual=0.3278 | target=0.3200 | diff=+0.0078
B: actual=0.3361 | target=0.3500 | diff=-0.0139


,Channel,Total candidates,Threshold skips,MAX_H skips,Overflow skips,Effective capacity,Effective %,Actual Ratio,Channel Target
0,G,196605,15478,14957,664,165506,84.18,0.3361,0.33
1,R,196605,15478,19107,573,161447,82.12,0.3278,0.32
2,B,196605,15478,14957,664,165506,84.18,0.3361,0.35
3,TOTAL,589815,46434,49021,1901,492459,83.49,1.0000,-


Total Capacity = 492,459 bit (480.92 Kbit)

Loaded image: Hand.jpg
Shape: (512, 512, 3)

=== CAPACITY vs CHANNEL TARGET ===
G: actual=0.3361 | target=0.3300 | diff=+0.0061
R: actual=0.3278 | target=0.3200 | diff=+0.0078
B: actual=0.3361 | target=0.3500 | diff=-0.0139


...

Total Capacity = 273,950 bit (267.53 Kbit)

Loaded image: sailboat.tiff
Shape: (512, 512, 3)

=== CAPACITY vs CHANNEL TARGET ===
G: actual=0.3123 | target=0.3300 | diff=-0.0177
R: actual=0.3031 | target=0.3200 | diff=-0.0169
B: actual=0.3846 | target=0.3500 | diff=+0.0346


,Channel,Total candidates,Threshold skips,MAX_H skips,Overflow skips,Effective capacity,Effective %,Actual Ratio,Channel Target
0,G,196605,36024,90815,110,69656,35.43,0.3123,0.33
1,R,196605,35589,93399,0,67617,34.39,0.3031,0.32
2,B,196605,34121,76630,56,85798,43.64,0.3846,0.35
3,TOTAL,589815,105734,260844,166,223071,37.82,1.0000,-


Total Capacity = 223,071 bit (217.84 Kbit)

Loaded image: splash.tiff
Shape: (512, 512, 3)

=== CAPACITY vs CHANNEL TARGET ===
G: actual=0.3000 | target=0.3300 | diff=-0.0300
R: actual=0.3579 | target=0.3200 | diff=+0.0379
B: actual=0.3422 | target=0.3500 | diff=-0.0078


,Channel,Total candidates,Threshold skips,MAX_H skips,Overflow skips,Effective capacity,Effective %,Actual Ratio,Channel Target
0,G,196605,29174,38007,392,129032,65.63,0.3000,0.33
1,R,196605,23221,19445,0,153939,78.30,0.3579,0.32
2,B,196605,27834,21565,0,147206,74.87,0.3422,0.35
3,TOTAL,589815,80229,79017,392,430177,72.93,1.0000,-


Total Capacity = 430,177 bit (420.09 Kbit)



In [11]:
# ============================================================
# COMPUTE AVERAGE CHANNEL RATIO (FROM CAPACITY ANALYSIS)
# ============================================================

def compute_avg_channel_ratio_from_capacity():

    print('\n' + '='*60)
    print('COMPUTE AVERAGE CHANNEL RATIO (CAPACITY-BASED)')
    print('='*60)

    G_list, R_list, B_list = [], [], []

    if len(IMAGE_FILES) == 0:
        print('[ERROR] IMAGE_FILES kosong')
        return None

    for img_name in IMAGE_FILES:

        img_path = os.path.join(DATASET_PATH, img_name)

        img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            print(f'[WARN] Gagal load: {img_name}')
            continue

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        reports, total_capacity = check_image_capacity(img_rgb)

        if total_capacity == 0:
            print(f'[WARN] Capacity nol: {img_name}')
            continue

        cap_r = reports['R']['effective_capacity']
        cap_g = reports['G']['effective_capacity']
        cap_b = reports['B']['effective_capacity']

        ratio_r = cap_r / total_capacity
        ratio_g = cap_g / total_capacity
        ratio_b = cap_b / total_capacity

        G_list.append(ratio_g)
        R_list.append(ratio_r)
        B_list.append(ratio_b)

        print(f'{img_name}: G={ratio_g:.4f} R={ratio_r:.4f} B={ratio_b:.4f}')

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------
    if len(G_list) == 0:
        print('[ERROR] Tidak ada data valid')
        return None

    # --------------------------------------------------------
    # MEAN
    # --------------------------------------------------------
    G_mean = np.mean(G_list)
    R_mean = np.mean(R_list)
    B_mean = np.mean(B_list)

    total = G_mean + R_mean + B_mean

    weights = {
        'G': G_mean / total,
        'R': R_mean / total,
        'B': B_mean / total
    }

    # --------------------------------------------------------
    # OUTPUT
    # --------------------------------------------------------
    print('\n=== FINAL AVERAGE WEIGHT (CAPACITY-BASED) ===')
    print(f"G: {weights['G']:.4f}")
    print(f"R: {weights['R']:.4f}")
    print(f"B: {weights['B']:.4f}")

    return weights

# ============================================================
# RUN + VALIDATION
# ============================================================

try:
    weights = compute_avg_channel_ratio_from_capacity()

    if weights is not None:
        print('\n[OK] Weight berhasil dihitung')
    else:
        print('\n[FAIL] Weight tidak tersedia')

except Exception as e:
    print('[ERROR]', e)


COMPUTE AVERAGE CHANNEL RATIO (CAPACITY-BASED)
Abdomal.jpg: G=0.3378 R=0.3244 B=0.3378
Chest.jpg: G=0.3361 R=0.3278 B=0.3361
Hand.jpg: G=0.3361 R=0.3278 B=0.3361
Head.jpg: G=0.3361 R=0.3277 B=0.3361
Leg.jpg: G=0.3354 R=0.3291 B=0.3354
airplane.tiff: G=0.3313 R=0.3148 B=0.3538
baboon.tiff: G=0.3321 R=0.3292 B=0.3387
house.tiff: G=0.3429 R=0.3037 B=0.3534
pepper.tiff: G=0.3617 R=0.2852 B=0.3531
sailboat.tiff: G=0.3123 R=0.3031 B=0.3846
splash.tiff: G=0.3000 R=0.3579 B=0.3422

=== FINAL AVERAGE WEIGHT (CAPACITY-BASED) ===
G: 0.3329
R: 0.3210
B: 0.3461

[OK] Weight berhasil dihitung


In [12]:
# ============================================================
# CELL 6: extract_channel
# DE Reversible + RM Map
# ============================================================

def extract_channel(
        stego_ch: np.ndarray,
        rm_map: np.ndarray,
        n_bits_target: int,
        ch_name: str = ''
    ):

    H, W = stego_ch.shape

    # --------------------------------------------------
    # flatten
    # --------------------------------------------------
    vec = stego_ch.flatten().astype(np.int32)

    rm_vec = rm_map.flatten().astype(np.uint8)

    # copy untuk restorasi cover
    cover_vec = vec.copy()

    total_px = H * W
    n_blocks = total_px // BLOCK_SIZE

    bits_out = []

    # --------------------------------------------------
    # proses block
    # --------------------------------------------------
    for b in range(n_blocks):

        start = b * BLOCK_SIZE
        end   = start + BLOCK_SIZE

        if end > len(vec):
            break

        # ----------------------------------------------
        # semua posisi embedding
        # ----------------------------------------------
        for pos in EMBED_POSITIONS:

            if len(bits_out) >= n_bits_target:
                break

            global_idx = start + pos

            if global_idx >= len(vec):
                continue

            # ------------------------------------------
            # hanya piksel yang benar-benar diembed
            # ------------------------------------------
            if rm_vec[global_idx] != 1:
                continue

            ref_local = get_ref_idx(pos)
            ref_idx   = start + ref_local

            if ref_idx >= len(vec):
                continue

            # ------------------------------------------
            # referensi harus berasal dari cover
            # yang sudah direstore
            # ------------------------------------------
            ref = int(cover_vec[ref_idx])

            stego_pixel = int(vec[global_idx])

            # ------------------------------------------
            # reverse DE
            # ------------------------------------------
            new_diff = stego_pixel - ref

            bit = new_diff & 1

            diff_orig = (new_diff - bit) // 2

            cover_pixel = ref + diff_orig

            # ------------------------------------------
            # restore pixel
            # ------------------------------------------
            cover_pixel = np.clip(
                cover_pixel,
                0,
                255
            )

            cover_vec[global_idx] = cover_pixel

            bits_out.append(int(bit))

        if len(bits_out) >= n_bits_target:
            break

    # --------------------------------------------------
    # reshape cover
    # --------------------------------------------------
    cover_ch = (
        cover_vec
        .reshape(H, W)
        .astype(np.uint8)
    )

    if ch_name:
        print(
            f'[{ch_name}] '
            f'Extracted {len(bits_out)}/{n_bits_target} bits'
        )

    return (
        bits_out[:n_bits_target],
        cover_ch
    )

In [13]:
# ============================================================
# CELL 7: Unit Test Reversibility
# ============================================================

def run_unit_tests():

    print('=== UNIT TEST REVERSIBILITY ===')

    all_pass = True

    # ======================================================
    # TEST 1
    # Manual DE
    # ======================================================
    print('\n[Test 1] Manual DE cases')

    manual_cases = [
        (0, 0, 100),
        (0, 1, 100),
        (3, 0, 100),
        (3, 1, 100),
        (-3, 0, 200),
        (-3, 1, 200)
    ]

    for diff, bit, ref in manual_cases:

        pixel = ref + diff

        new_diff = diff * 2 + bit

        stego_pixel = ref + new_diff

        nd2 = stego_pixel - ref

        bit_rec = nd2 & 1

        diff_rec = (nd2 - bit_rec) // 2

        pixel_rec = ref + diff_rec

        ok = (
            bit_rec == bit and
            pixel_rec == pixel
        )

        print(
            f'  {"✓" if ok else "✗"} '
            f'diff={diff:+d} bit={bit}'
        )

        all_pass &= ok

    # ======================================================
    # TEST 2
    # Random Channel
    # ======================================================
    print('\n[Test 2] Random Channel')

    try:

        np.random.seed(7)

        test_ch = np.random.randint(
            20,
            230,
            (64, 64),
            dtype=np.uint8
        )

        rng = np.random.default_rng(99)

        test_bits = rng.integers(
            0,
            2,
            5000
        ).tolist()

        stego_ch, rm_map, n_emb, block_meta = embed_channel(
            test_ch.astype(np.int32),
            test_bits,
            'G'
        )

        ext_bits, cover_rec = extract_channel(
            stego_ch,
            rm_map,
            n_emb,
            'G'
        )

        orig_bits = test_bits[:n_emb]

        ber = (
            sum(
                a != b
                for a, b in zip(orig_bits, ext_bits)
            )
            / max(n_emb, 1)
        )

        px_err = int(
            np.sum(
                test_ch != cover_rec
            )
        )

        print(f'  Embedded bits : {n_emb}')
        print(f'  BER           : {ber:.6f}')
        print(f'  Pixel errors  : {px_err}')

        ok2 = (
            ber == 0.0 and
            px_err == 0 and
            n_emb > 0
        )

        print(
            f'  {"✓ LULUS" if ok2 else "✗ GAGAL"}'
        )

        all_pass &= ok2

    except Exception as e:

        print(f'  ✗ ERROR : {e}')
        all_pass = False

    # ======================================================
    # TEST 3
    # Block Consistency
    # ======================================================
    print('\n[Test 3] Block Consistency')

    try:

        test_block = np.array([
            100,102,101,103,
            150,149,151,152,
            200,198,201,199
        ], dtype=np.uint8).reshape(3,4)

        test_ch2 = np.tile(
            test_block,
            (16,16)
        )

        rng = np.random.default_rng(123)

        test_bits2 = rng.integers(
            0,
            2,
            1000
        ).tolist()

        stego_ch2, rm_map2, n_emb2, block_meta2 = embed_channel(
            test_ch2.astype(np.int32),
            test_bits2,
            'G'
        )

        ext_bits2, cover_rec2 = extract_channel(
            stego_ch2,
            rm_map2,
            n_emb2,
            'G'
        )

        orig_bits2 = test_bits2[:n_emb2]

        ber2 = (
            sum(
                a != b
                for a, b in zip(orig_bits2, ext_bits2)
            )
            / max(n_emb2, 1)
        )

        px_err2 = int(
            np.sum(
                test_ch2 != cover_rec2
            )
        )

        print(f'  Embedded bits : {n_emb2}')
        print(f'  BER           : {ber2:.6f}')
        print(f'  Pixel errors  : {px_err2}')

        ok3 = (
            ber2 == 0.0 and
            px_err2 == 0 and
            n_emb2 > 0
        )

        print(
            f'  {"✓ LULUS" if ok3 else "✗ GAGAL"}'
        )

        all_pass &= ok3

    except Exception as e:

        print(f'  ✗ ERROR : {e}')
        all_pass = False

    # ======================================================
    # FINAL RESULT
    # ======================================================
    print('\n' + '=' * 60)

    if all_pass:
        print('FINAL RESULT : ✓ SEMUA TEST LULUS')
    else:
        print('FINAL RESULT : ✗ ADA TEST YANG GAGAL')

    print('=' * 60)

    return all_pass


# ============================================================
# RUN TEST
# ============================================================

assert run_unit_tests(), (
    'Unit test gagal. '
    'Periksa Cell 5 atau Cell 6.'
)

=== UNIT TEST REVERSIBILITY ===

[Test 1] Manual DE cases
  ✓ diff=+0 bit=0
  ✓ diff=+0 bit=1
  ✓ diff=+3 bit=0
  ✓ diff=+3 bit=1
  ✓ diff=-3 bit=0
  ✓ diff=-3 bit=1

[Test 2] Random Channel
[G] Embedded 35/5000 bits  (0.70%) 
[G] Extracted 35/35 bits
  Embedded bits : 35
  BER           : 0.000000
  Pixel errors  : 0
  ✓ LULUS

[Test 3] Block Consistency
[G] Embedded 1000/1000 bits  (100.00%) 
[G] Extracted 1000/1000 bits
  Embedded bits : 1000
  BER           : 0.000000
  Pixel errors  : 0
  ✓ LULUS

FINAL RESULT : ✓ SEMUA TEST LULUS


In [14]:
# ============================================================
# CELL 8: embed_image (REVISED - SAFE & SYNC)
# ============================================================

def embed_image(
        cover_path: str,
        bits_all: list,
        output_path: str
    ) -> dict:

    print(f'\n[EMBEDDING] {os.path.basename(cover_path)}')

    # =====================================================
    # LOAD IMAGE
    # =====================================================
    cover_bgr = cv2.imread(cover_path, cv2.IMREAD_COLOR)

    if cover_bgr is None:
        raise FileNotFoundError(f'Cover tidak ditemukan:\n{cover_path}')

    img = cv2.cvtColor(cover_bgr, cv2.COLOR_BGR2RGB)

    H, W = img.shape[:2]

    print(f'  Ukuran      : {W}x{H}')
    print(f'  Payload     : {len(bits_all):,} bit')

    # =====================================================
    # SPLIT CHANNEL
    # =====================================================
    R_ch = img[:, :, 0].astype(np.int32)
    G_ch = img[:, :, 1].astype(np.int32)
    B_ch = img[:, :, 2].astype(np.int32)

    total_bits = len(bits_all)

    # =====================================================
    # CHANNEL ALLOCATION
    # =====================================================
    n_G_target = int(total_bits * CHANNEL_WEIGHT['G'])
    n_R_target = int(total_bits * CHANNEL_WEIGHT['R'])
    n_B_target = total_bits - n_G_target - n_R_target

    bits_G = bits_all[:n_G_target]
    bits_R = bits_all[n_G_target:n_G_target + n_R_target]
    bits_B = bits_all[n_G_target + n_R_target:]

    print(f'  Target G/R/B: {n_G_target:,} / {n_R_target:,} / {n_B_target:,}')

    # =====================================================
    # EMBEDDING PER CHANNEL
    # =====================================================
    G_stego, G_rm, G_nb, G_meta = embed_channel(G_ch, bits_G, 'G')
    R_stego, R_rm, R_nb, R_meta = embed_channel(R_ch, bits_R, 'R')
    B_stego, B_rm, B_nb, B_meta = embed_channel(B_ch, bits_B, 'B')

    total_embedded = G_nb + R_nb + B_nb

    print(f'  Embedded G : {G_nb:,}')
    print(f'  Embedded R : {R_nb:,}')
    print(f'  Embedded B : {B_nb:,}')
    print(f'  Total      : {total_embedded:,}/{total_bits:,}')

    if total_embedded < total_bits:
        print(f'  [WARNING] {total_bits-total_embedded:,} bit tidak berhasil diembed')

    # =====================================================
    # MERGE RGB
    # =====================================================
    stego_rgb = np.stack([R_stego, G_stego, B_stego], axis=2)

    # ⚠️ FIX: clip untuk hindari overflow
    stego_rgb = np.clip(stego_rgb, 0, 255).astype(np.uint8)

    stego_bgr = cv2.cvtColor(stego_rgb, cv2.COLOR_RGB2BGR)
    cv2.imwrite(output_path, stego_bgr)

    print(f'  Stego saved : {output_path}')

    # =====================================================
    # METRICS (COVER vs STEGO) ✅ SUDAH BENAR
    # =====================================================
    cover_f = img.astype(np.float64)
    stego_f = stego_rgb.astype(np.float64)

    diff = cover_f - stego_f
    mse_val = float(np.mean(diff ** 2))

    if mse_val > 1e-12:
        psnr_val = float(10 * np.log10((255.0 ** 2) / mse_val))
    else:
        psnr_val = 999.0

    ssim_val = float(
        ssim_fn(
            cover_f,
            stego_f,
            channel_axis=2,
            data_range=255
        )
    )

    print(f'  PSNR = {psnr_val:.4f} dB')
    print(f'  SSIM = {ssim_val:.6f}')
    print(f'  MSE  = {mse_val:.6f}')

    # =====================================================
    # PER-CHANNEL METRICS
    # =====================================================
    mse_r = float(np.mean((cover_f[:, :, 0] - stego_f[:, :, 0]) ** 2))
    mse_g = float(np.mean((cover_f[:, :, 1] - stego_f[:, :, 1]) ** 2))
    mse_b = float(np.mean((cover_f[:, :, 2] - stego_f[:, :, 2]) ** 2))

    def calc_psnr(mse):
        return float(10 * np.log10(255**2 / mse)) if mse > 1e-12 else 999.0

    psnr_r = calc_psnr(mse_r)
    psnr_g = calc_psnr(mse_g)
    psnr_b = calc_psnr(mse_b)

    ssim_r = float(ssim_fn(cover_f[:, :, 0], stego_f[:, :, 0], data_range=255))
    ssim_g = float(ssim_fn(cover_f[:, :, 1], stego_f[:, :, 1], data_range=255))
    ssim_b = float(ssim_fn(cover_f[:, :, 2], stego_f[:, :, 2], data_range=255))

    # =====================================================
    # METADATA
    # =====================================================
    metadata = {
        'image_shape': [H, W],
        'total_bits': total_bits,
        'embedded_bits': total_embedded,

        'embedded_G': G_nb,
        'embedded_R': R_nb,
        'embedded_B': B_nb,

        'channel_weight': CHANNEL_WEIGHT,
        'max_h': MAX_H,
        'threshold_mode': THRESHOLD_MODE,

        'G_rm': G_rm.flatten().tolist(),
        'R_rm': R_rm.flatten().tolist(),
        'B_rm': B_rm.flatten().tolist(),

        'G_block_meta': G_meta,
        'R_block_meta': R_meta,
        'B_block_meta': B_meta,

        'metrics': {
            'PSNR': psnr_val,
            'SSIM': ssim_val,
            'MSE': mse_val,

            'PSNR_R': psnr_r,
            'PSNR_G': psnr_g,
            'PSNR_B': psnr_b,

            'SSIM_R': ssim_r,
            'SSIM_G': ssim_g,
            'SSIM_B': ssim_b,

            'MSE_R': mse_r,
            'MSE_G': mse_g,
            'MSE_B': mse_b
        }
    }

    # =====================================================
    # SAVE JSON
    # =====================================================
    base_name = os.path.splitext(output_path)[0]
    meta_path = base_name + '_meta.json'

    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2)

    print(f'  Metadata : {meta_path}')

    return metadata


print('embed_image defined (REVISED).')

embed_image defined (REVISED).


In [15]:
# ============================================================
# CELL 9: extract_image
# ============================================================

def extract_image(
        stego_path: str,
        metadata: dict
    ):

    print('\n[EXTRACTION + RECOVERY]')

    # --------------------------------------------------
    # metadata validation
    # --------------------------------------------------
    if metadata.get('threshold_mode') != THRESHOLD_MODE:
        print(
            f'[WARN] threshold_mode mismatch '
            f'({metadata.get("threshold_mode")} != {THRESHOLD_MODE})'
        )

    if metadata.get('max_h') != MAX_H:
        print('[WARN] MAX_H berbeda dengan config')

    if metadata.get('channel_weight') != CHANNEL_WEIGHT:
        print('[WARN] CHANNEL_WEIGHT berbeda dengan config')

    # --------------------------------------------------
    # load stego image
    # --------------------------------------------------
    stego_bgr = cv2.imread(
        stego_path,
        cv2.IMREAD_COLOR
    )

    if stego_bgr is None:
        raise FileNotFoundError(
            f'Stego image tidak ditemukan:\n{stego_path}'
        )

    stego_rgb = cv2.cvtColor(
        stego_bgr,
        cv2.COLOR_BGR2RGB
    )

    # --------------------------------------------------
    # image shape
    # --------------------------------------------------
    H, W = metadata['image_shape']

    if stego_rgb.shape[:2] != (H, W):
        raise ValueError(
            f'Image size mismatch. '
            f'Expected {(H,W)}, got {stego_rgb.shape[:2]}'
        )

    print(f'Image size : {W}x{H}')

    # --------------------------------------------------
    # RM MAP
    # --------------------------------------------------
    G_rm = np.array(
        metadata['G_rm'],
        dtype=np.uint8
    ).reshape(H, W)

    R_rm = np.array(
        metadata['R_rm'],
        dtype=np.uint8
    ).reshape(H, W)

    B_rm = np.array(
        metadata['B_rm'],
        dtype=np.uint8
    ).reshape(H, W)

    # --------------------------------------------------
    # split channel
    # --------------------------------------------------
    R_ch = stego_rgb[:, :, 0]
    G_ch = stego_rgb[:, :, 1]
    B_ch = stego_rgb[:, :, 2]

    # --------------------------------------------------
    # embedded bit count
    # --------------------------------------------------
    n_G = int(metadata['embedded_G'])
    n_R = int(metadata['embedded_R'])
    n_B = int(metadata['embedded_B'])

    total_expected = int(
        metadata.get(
            'embedded_bits',
            n_G + n_R + n_B
        )
    )

    print(
        f'Expected bits: '
        f'G={n_G:,}  '
        f'R={n_R:,}  '
        f'B={n_B:,}'
    )

    # --------------------------------------------------
    # extraction
    # --------------------------------------------------
    bits_G, cover_G = extract_channel(
        G_ch,
        G_rm,
        n_G,
        'G'
    )

    bits_R, cover_R = extract_channel(
        R_ch,
        R_rm,
        n_R,
        'R'
    )

    bits_B, cover_B = extract_channel(
        B_ch,
        B_rm,
        n_B,
        'B'
    )

    # --------------------------------------------------
    # rebuild payload
    # --------------------------------------------------
    extracted_bits = (
        bits_G +
        bits_R +
        bits_B
    )

    if len(extracted_bits) != total_expected:

        raise ValueError(
            f'Extracted bits mismatch: '
            f'{len(extracted_bits)} != {total_expected}'
        )

    # --------------------------------------------------
    # reconstruct cover image
    # --------------------------------------------------
    recovered_rgb = np.stack(
        [
            cover_R,
            cover_G,
            cover_B
        ],
        axis=2
    ).astype(np.uint8)

    # --------------------------------------------------
    # statistics
    # --------------------------------------------------
    print('\n=== EXTRACTION REPORT ===')

    print(
        f'G : {len(bits_G):,} bits'
    )

    print(
        f'R : {len(bits_R):,} bits'
    )

    print(
        f'B : {len(bits_B):,} bits'
    )

    print(
        f'Total extracted : '
        f'{len(extracted_bits):,} bits'
    )

    print(
        f'Recovered image : '
        f'{recovered_rgb.shape}'
    )

    # --------------------------------------------------
    # return
    # --------------------------------------------------
    return (
        extracted_bits,
        recovered_rgb
    )


print('extract_image defined.')

extract_image defined.


In [ ]:
# # ============================================================
# # CHECK ALL NUMPY VARIABLES (SAFE VERSION)
# # ============================================================

# all_vars = list(globals().items())

# for var_name, var_value in all_vars:
#     try:
#         if isinstance(var_value, np.ndarray):
#             print(var_name, var_value.shape, var_value.dtype)
#     except:
#         pass

cover_img (512, 512, 3) uint8
B (512, 512) uint8
G (512, 512) uint8
R (512, 512) uint8
payload_bits (1000,) int32
stego_G (512, 512) uint8
rm_map_G (512, 512) uint8
stego_bgr (512, 512, 3) uint8


In [16]:
# ============================================================
# CELL 10: Visualization
# ============================================================

def save_visual(
        cover_path,
        stego_path,
        restored_rgb,
        payload_kbit,
        out_dir
    ):
    """
    Save:
        - cover
        - stego
        - restored
        - amplified difference map

    + reversible recovery verification
    """

    # --------------------------------------------------
    # load images
    # --------------------------------------------------
    cover_bgr = cv2.imread(
        cover_path,
        cv2.IMREAD_COLOR
    )

    stego_bgr = cv2.imread(
        stego_path,
        cv2.IMREAD_COLOR
    )

    if cover_bgr is None:
        raise FileNotFoundError(
            f'Cover image not found:\n{cover_path}'
        )

    if stego_bgr is None:
        raise FileNotFoundError(
            f'Stego image not found:\n{stego_path}'
        )

    cover_rgb = cv2.cvtColor(
        cover_bgr,
        cv2.COLOR_BGR2RGB
    )

    stego_rgb = cv2.cvtColor(
        stego_bgr,
        cv2.COLOR_BGR2RGB
    )

    # --------------------------------------------------
    # shape validation
    # --------------------------------------------------
    if cover_rgb.shape != stego_rgb.shape:
        raise ValueError(
            f'Shape mismatch:\n'
            f'Cover={cover_rgb.shape}\n'
            f'Stego={stego_rgb.shape}'
        )

    if cover_rgb.shape != restored_rgb.shape:
        raise ValueError(
            f'Shape mismatch:\n'
            f'Cover={cover_rgb.shape}\n'
            f'Restored={restored_rgb.shape}'
        )

    # --------------------------------------------------
    # difference map
    # --------------------------------------------------
    diff_rgb = np.abs(
        cover_rgb.astype(np.int32)
        -
        stego_rgb.astype(np.int32)
    )

    diff_gray = np.max(
        diff_rgb,
        axis=2
    ).astype(np.float32)

    # --------------------------------------------------
    # diagnostics
    # --------------------------------------------------
    changed_pixels = int(
        np.sum(diff_gray > 0)
    )

    total_pixels = diff_gray.size

    pct_changed = (
        100 * changed_pixels /
        max(total_pixels, 1)
    )

    max_diff = float(diff_gray.max())

    print(
        f'    Changed pixels : '
        f'{changed_pixels:,}/{total_pixels:,} '
        f'({pct_changed:.2f}%)'
    )

    print(
        f'    Max difference : '
        f'{max_diff:.2f}'
    )

    # --------------------------------------------------
    # dynamic amplification
    # --------------------------------------------------
    nonzero = diff_gray[diff_gray > 0]

    if len(nonzero) > 0:

        p95 = float(
            np.percentile(
                nonzero,
                95
            )
        )

        amp_factor = min(
            max(
                1,
                int(180 / max(p95, 1e-6))
            ),
            64
        )

    else:
        amp_factor = 1

    diff_vis = np.clip(
        diff_gray * amp_factor,
        0,
        255
    ).astype(np.uint8)

    print(
        f'    Amplification : '
        f'{amp_factor}x'
    )

    # --------------------------------------------------
    # reversible recovery check
    # --------------------------------------------------
    pixel_error = int(
        np.sum(
            np.any(
                cover_rgb != restored_rgb,
                axis=2
            )
        )
    )

    if pixel_error == 0:
        print(
            '    Recovery : PERFECT'
        )
    else:
        print(
            f'    Recovery : '
            f'{pixel_error} pixel mismatch'
        )

    # --------------------------------------------------
    # output paths
    # --------------------------------------------------
    os.makedirs(
        out_dir,
        exist_ok=True
    )

    base_name = os.path.splitext(
        os.path.basename(cover_path)
    )[0]

    restored_path = os.path.join(
        out_dir,
        f'{base_name}_restored_{payload_kbit}kbit.png'
    )

    diff_path = os.path.join(
        out_dir,
        f'{base_name}_diff_{payload_kbit}kbit.png'
    )

    figure_path = os.path.join(
        out_dir,
        f'{base_name}_comparison_{payload_kbit}kbit.png'
    )

    # --------------------------------------------------
    # save restored
    # --------------------------------------------------
    cv2.imwrite(
        restored_path,
        cv2.cvtColor(
            restored_rgb,
            cv2.COLOR_RGB2BGR
        )
    )

    cv2.imwrite(
        diff_path,
        diff_vis
    )

    # --------------------------------------------------
    # visualization
    # --------------------------------------------------
    fig, ax = plt.subplots(
        2,
        2,
        figsize=(12,10)
    )

    fig.suptitle(
        f'{base_name} | '
        f'{payload_kbit} Kbit',
        fontsize=13,
        fontweight='bold'
    )

    ax[0,0].imshow(cover_rgb)
    ax[0,0].set_title('Cover')
    ax[0,0].axis('off')

    ax[0,1].imshow(stego_rgb)
    ax[0,1].set_title('Stego')
    ax[0,1].axis('off')

    ax[1,0].imshow(restored_rgb)
    ax[1,0].set_title(
        f'Restored (err={pixel_error})'
    )
    ax[1,0].axis('off')

    im = ax[1,1].imshow(
        diff_vis,
        cmap='hot',
        vmin=0,
        vmax=255
    )

    ax[1,1].set_title(
        f'Difference ×{amp_factor}'
    )
    ax[1,1].axis('off')

    plt.colorbar(
        im,
        ax=ax[1,1],
        fraction=0.046,
        pad=0.04
    )

    plt.tight_layout()

    plt.savefig(
        figure_path,
        dpi=150,
        bbox_inches='tight'
    )

    plt.close()

    print(
        f'    Saved : '
        f'{os.path.basename(figure_path)}'
    )

    return {
        'pixel_error': pixel_error,
        'changed_pixels': changed_pixels,
        'change_ratio':
            changed_pixels / max(total_pixels, 1),
        'amplification': amp_factor,
        'figure_path': figure_path,
        'restored_path': restored_path,
        'diff_path': diff_path
    }


print('save_visual defined.')

save_visual defined.


In [17]:
# ============================================================
# CELL 11: CSV + BER
# ============================================================

import csv
import numpy as np

# ============================================================
# BER
# ============================================================

def compute_ber(
    orig_bits: list,
    recv_bits: list
) -> float:

    n_orig = len(orig_bits)

    if n_orig == 0:
        return 0.0

    n_cmp = min(
        len(orig_bits),
        len(recv_bits)
    )

    errors = sum(
        a != b
        for a, b in zip(
            orig_bits[:n_cmp],
            recv_bits[:n_cmp]
        )
    )

    errors += abs(
        len(orig_bits) -
        len(recv_bits)
    )

    return errors / n_orig


# ============================================================
# BUILD RESULT ROW
# ============================================================

def build_result_row(
    image_name: str,
    payload_kbit: int,
    metadata: dict,
    bits_original: list,
    bits_extracted: list,
    cover_rgb: np.ndarray,
    restored_rgb: np.ndarray
):

    # --------------------------------------------------------
    # BASIC COUNTS
    # --------------------------------------------------------
    bits_target = len(bits_original)

    bits_embedded = int(
        metadata.get(
            'embedded_bits',
            0
        )
    )

    bits_extracted_n = len(bits_extracted)

    # --------------------------------------------------------
    # BER
    # --------------------------------------------------------
    ber = compute_ber(
        bits_original[:bits_embedded],
        bits_extracted[:bits_embedded]
    )

    # --------------------------------------------------------
    # MESSAGE MATCH
    # --------------------------------------------------------
    n_cmp = min(
        bits_embedded,
        len(bits_extracted)
    )

    msg_match = (
        bits_original[:n_cmp]
        ==
        bits_extracted[:n_cmp]
    )

    # --------------------------------------------------------
    # COVER RECOVERY
    # --------------------------------------------------------
    diff = np.abs(
        cover_rgb.astype(np.int32)
        -
        restored_rgb.astype(np.int32)
    )

    px_err = int(
        np.sum(
            np.any(diff > 0, axis=2)
        )
    )

    max_diff = int(diff.max())

    cover_exact = (px_err == 0)

    # --------------------------------------------------------
    # UTILIZATION
    # --------------------------------------------------------
    utilization = (
        bits_embedded / bits_target
        if bits_target > 0
        else 0.0
    )

    # --------------------------------------------------------
    # RESULT ROW
    # --------------------------------------------------------
    row = {

        # identity
        'Image': image_name,
        'Payload_Kbit': payload_kbit,

        # global metrics
        'PSNR': metadata['metrics']['PSNR'],
        'SSIM': metadata['metrics']['SSIM'],
        'MSE': metadata['metrics']['MSE'],

        # channel metrics
        'PSNR_R': metadata['metrics']['PSNR_R'],
        'PSNR_G': metadata['metrics']['PSNR_G'],
        'PSNR_B': metadata['metrics']['PSNR_B'],

        'SSIM_R': metadata['metrics']['SSIM_R'],
        'SSIM_G': metadata['metrics']['SSIM_G'],
        'SSIM_B': metadata['metrics']['SSIM_B'],

        'MSE_R': metadata['metrics']['MSE_R'],
        'MSE_G': metadata['metrics']['MSE_G'],
        'MSE_B': metadata['metrics']['MSE_B'],

        # payload stats
        'Bits_Target': bits_target,
        'Bits_Embedded': bits_embedded,
        'Bits_Extracted': bits_extracted_n,

        'Embedded_G': metadata['embedded_G'],
        'Embedded_R': metadata['embedded_R'],
        'Embedded_B': metadata['embedded_B'],

        # config
        'Threshold_Mode':
            metadata.get(
                'threshold_mode',
                'unknown'
            ),

        # payload correctness
        'BER': ber,
        'Message_Match': int(msg_match),

        # reversible recovery
        'Cover_Exact': int(cover_exact),
        'Recovery_Perfect': int(cover_exact),

        'Cover_Pixel_Error': px_err,
        'Cover_Max_Diff': max_diff,

        # utilization
        'Utilization': utilization
    }

    return row


# ============================================================
# SAVE CSV
# ============================================================

def save_csv(
    results: list,
    csv_path: str
):

    if len(results) == 0:
        print('[WARN] Tidak ada data untuk disimpan')
        return

    fields = list(results[0].keys())

    with open(
        csv_path,
        'w',
        newline='',
        encoding='utf-8'
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fields
        )

        writer.writeheader()

        for row in results:

            clean_row = {}

            for k, v in row.items():

                if isinstance(v, float):
                    clean_row[k] = round(v, 6)
                else:
                    clean_row[k] = v

            writer.writerow(clean_row)

    print(
        f'[CSV SAVED] {csv_path}'
    )


print(
    'CSV utilities defined '
)

CSV utilities defined 


In [19]:
# ============================================================
# CELL DEBUG: FINAL CONSISTENCY CHECK
# ============================================================

def check_final_consistency(orig_bits, recv_bits, cover_rgb, restored_rgb, n_emb):

    print('\n=== FINAL CONSISTENCY CHECK ===')

    if n_emb == 0:
        print('[WARN] n_emb = 0, tidak bisa compare')
        return False

    # -----------------------------
    # BIT CHECK
    # -----------------------------
    orig_cmp = orig_bits[:n_emb]
    recv_cmp = recv_bits[:n_emb]

    ber = compute_ber(orig_cmp, recv_cmp)

    # -----------------------------
    # PIXEL CHECK
    # -----------------------------
    diff = np.abs(
        cover_rgb.astype(np.int32) -
        restored_rgb.astype(np.int32)
    )

    pixel_errors = int(np.sum(np.any(diff > 0, axis=2)))
    max_diff     = int(diff.max())

    # -----------------------------
    # FLAGS
    # -----------------------------
    msg_ok   = (ber == 0.0)
    cover_ok = (pixel_errors == 0 and max_diff == 0)

    # -----------------------------
    # OUTPUT
    # -----------------------------
    print(f'BER           : {ber:.10f}')
    print(f'Pixel Errors  : {pixel_errors}')
    print(f'Max Diff      : {max_diff}')
    print(f'Message Match : {msg_ok}')
    print(f'Cover Exact   : {cover_ok}')

    if msg_ok and cover_ok:
        print('\n✓ PERFECT REVERSIBILITY')
    else:
        print('\n✗ NOT REVERSIBLE')

    return msg_ok and cover_ok


print('[INFO] check_final_consistency READY')

[INFO] check_final_consistency READY


In [20]:
print("\n=== DEBUG METADATA ===")

print(meta.keys())

print("\nMetrics keys:")
print(meta['metrics'].keys())

print("\nMetrics content:")
from pprint import pprint
pprint(meta['metrics'])


=== DEBUG METADATA ===
dict_keys(['image_shape', 'total_bits', 'embedded_bits', 'embedded_G', 'embedded_R', 'embedded_B', 'channel_weight', 'max_h', 'threshold_mode', 'G_rm', 'R_rm', 'B_rm', 'G_block_meta', 'R_block_meta', 'B_block_meta', 'metrics'])

Metrics keys:
dict_keys(['PSNR', 'SSIM', 'MSE', 'PSNR_R', 'PSNR_G', 'PSNR_B', 'SSIM_R', 'SSIM_G', 'SSIM_B', 'MSE_R', 'MSE_G', 'MSE_B'])

Metrics content:
{'MSE': 0.4287160237630208,
 'MSE_B': 0.49365234375,
 'MSE_G': 0.46076202392578125,
 'MSE_R': 0.33173370361328125,
 'PSNR': 51.80910644789499,
 'PSNR_B': 51.19659157580703,
 'PSNR_G': 51.496036835932244,
 'PSNR_R': 52.922907634826444,
 'SSIM': 0.9962435806390131,
 'SSIM_B': 0.995888534758697,
 'SSIM_G': 0.9961375430260037,
 'SSIM_R': 0.9967046641323383}


In [21]:
required = [
    'PSNR_R','PSNR_G','PSNR_B',
    'SSIM_R','SSIM_G','SSIM_B',
    'MSE_R','MSE_G','MSE_B'
]

for k in required:
    print(k, "->", k in meta['metrics'])

PSNR_R -> True
PSNR_G -> True
PSNR_B -> True
SSIM_R -> True
SSIM_G -> True
SSIM_B -> True
MSE_R -> True
MSE_G -> True
MSE_B -> True


In [23]:
# ============================================================
# QUICK TEST DEBUG (COMPACT) sebelum eksperimen utama
# ============================================================

print('='*60)
print('QUICK TEST DEBUG')
print('='*60)

# ------------------------------------------------------------
# PILIH 1 IMAGE
# ------------------------------------------------------------
COVER_FILE = IMAGE_FILES[0]

COVER_PATH = os.path.join(
    DATASET_PATH,
    COVER_FILE
)

PAYLOAD_KBIT = 70

print(f'Image   : {COVER_FILE}')
print(f'Payload : {PAYLOAD_KBIT} Kbit')

# ------------------------------------------------------------
# LOAD COVER
# ------------------------------------------------------------
cover_bgr = cv2.imread(
    COVER_PATH,
    cv2.IMREAD_COLOR
)

cover_rgb = cv2.cvtColor(
    cover_bgr,
    cv2.COLOR_BGR2RGB
)

# ------------------------------------------------------------
# LOAD PAYLOAD
# ------------------------------------------------------------
orig_bits = generate_bits_for_payload(
    PAYLOAD_KBIT
)

print(f'Bits loaded : {len(orig_bits):,}')

# ------------------------------------------------------------
# OUTPUT FILE
# ------------------------------------------------------------
img_base = os.path.splitext(
    COVER_FILE
)[0]

stego_path = os.path.join(
    OUTPUT_PATH,
    f'{img_base}_quicktest.png'
)

# ------------------------------------------------------------
# EMBEDDING
# ------------------------------------------------------------
meta = embed_image(
    COVER_PATH,
    orig_bits,
    stego_path
)

n_emb = meta['embedded_bits']

print(f'\nEmbedded : {n_emb:,} bits')

if n_emb == 0:
    raise RuntimeError(
        'Tidak ada bit yang berhasil diembed'
    )

# ------------------------------------------------------------
# EXTRACTION
# ------------------------------------------------------------
recv_bits, restored_rgb = extract_image(
    stego_path,
    meta
)

# ------------------------------------------------------------
# BER
# ------------------------------------------------------------
ber = compute_ber(
    orig_bits[:n_emb],
    recv_bits
)

# ------------------------------------------------------------
# RECOVERY CHECK
# ------------------------------------------------------------
diff = np.abs(
    cover_rgb.astype(np.int32)
    -
    restored_rgb.astype(np.int32)
)

pixel_errors = int(
    np.sum(
        np.any(diff > 0, axis=2)
    )
)

max_diff = int(diff.max())

# ------------------------------------------------------------
# RESULT
# ------------------------------------------------------------
print('\n=== RESULT ===')

print(f'BER           : {ber:.10f}')
print(f'Pixel Errors  : {pixel_errors}')
print(f'Max Diff      : {max_diff}')

msg_ok   = (ber == 0.0)
cover_ok = (pixel_errors == 0 and max_diff == 0)

print(f'Message Match : {msg_ok}')
print(f'Cover Exact   : {cover_ok}')

# ------------------------------------------------------------
# VISUALIZATION
# ------------------------------------------------------------
save_visual(
    COVER_PATH,
    stego_path,
    restored_rgb,
    PAYLOAD_KBIT,
    OUTPUT_PATH
)

# ------------------------------------------------------------
# FINAL CONSISTENCY CHECK (DEBUG TAMBAHAN)
# ------------------------------------------------------------
print('\n[DEBUG] masuk ke check_final_consistency...')

if 'check_final_consistency' in globals():
    check_final_consistency(
        orig_bits,
        recv_bits,
        cover_rgb,
        restored_rgb,
        n_emb
    )
else:
    print('[WARN] check_final_consistency belum didefinisikan')
    
# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------
if msg_ok and cover_ok:
    print('\nSUCCESS')
    print('Pipeline siap untuk eksperimen utama')
else:
    raise RuntimeError(
        '\nFAILED\n'
        'Masih ada masalah pada embedding/extraction'
    )

QUICK TEST DEBUG
Image   : Abdomal.jpg
Payload : 70 Kbit
Payload loaded: random-binary_70Kb.txt (70000 bit)
Bits loaded : 70,000

[EMBEDDING] Abdomal.jpg
  Ukuran      : 512x512
  Payload     : 70,000 bit
  Target G/R/B: 23,100 / 22,400 / 24,500
[G] Embedded 23100/23100 bits  (100.00%) 
[R] Embedded 22400/22400 bits  (100.00%) 
[B] Embedded 24500/24500 bits  (100.00%) 
  Embedded G : 23,100
  Embedded R : 22,400
  Embedded B : 24,500
  Total      : 70,000/70,000
  Stego saved : D:\KBJ-Komputasi Jaringan\data hiding paper\propose fix\output_p1090\Abdomal_quicktest.png
  PSNR = 51.8091 dB
  SSIM = 0.996244
  MSE  = 0.428716
  Metadata : D:\KBJ-Komputasi Jaringan\data hiding paper\propose fix\output_p1090\Abdomal_quicktest_meta.json

Embedded : 70,000 bits

[EXTRACTION + RECOVERY]
Image size : 512x512
Expected bits: G=23,100  R=22,400  B=24,500
[G] Extracted 23100/23100 bits
[R] Extracted 22400/22400 bits
[B] Extracted 24500/24500 bits

=== EXTRACTION REPORT ===
G : 23,100 bits
R : 22,400

In [24]:
# ============================================================
# CELL 12: MAIN EXPERIMENT LOOP (MULTI IMAGE)
# ============================================================

print('=' * 80)
print('MULTI IMAGE EVALUATION')
print('=' * 80)

image_files = sorted([
    f for f in os.listdir(DATASET_PATH)
    if f.lower().endswith(SUPPORTED_EXTENSIONS)
])

print(f'Total images : {len(image_files)}')

all_results = []

# ============================================================
# IMAGE LOOP
# ============================================================

for image_name in image_files:

    print('\n' + '=' * 80)
    print(f'IMAGE : {image_name}')
    print('=' * 80)

    cover_path = os.path.join(
        DATASET_PATH,
        image_name
    )

    try:

        cover_bgr = cv2.imread(
            cover_path,
            cv2.IMREAD_COLOR
        )

        if cover_bgr is None:
            raise ValueError(
                f'Cannot read image: {image_name}'
            )

        cover_rgb = cv2.cvtColor(
            cover_bgr,
            cv2.COLOR_BGR2RGB
        )

    except Exception as e:

        print(f'[SKIP] {image_name}: {e}')
        continue

    # ========================================================
    # PAYLOAD LOOP
    # ========================================================

    for payload_kbit in PAYLOAD_SIZES_KBIT:

        print(
            f'\n[{image_name}] '
            f'Payload {payload_kbit} Kbit'
        )

        try:

            # ------------------------------------------------
            # LOAD PAYLOAD
            # ------------------------------------------------
            orig_bits = generate_bits_for_payload(
                payload_kbit
            )

            img_base = os.path.splitext(
                image_name
            )[0]

            stego_path = os.path.join(
                OUTPUT_PATH,
                f'{img_base}_stego_{payload_kbit}kbit.png'
            )

            # ------------------------------------------------
            # EMBEDDING
            # ------------------------------------------------
            meta = embed_image(
                cover_path,
                orig_bits,
                stego_path
            )

            n_emb = meta['embedded_bits']

            # ------------------------------------------------
            # EXTRACTION
            # ------------------------------------------------
            recv_bits, restored_rgb = extract_image(
                stego_path,
                meta
            )

            # ------------------------------------------------
            # BER
            # ------------------------------------------------
            ber = compute_ber(
                orig_bits[:n_emb],
                recv_bits
            )

            # ------------------------------------------------
            # VISUALIZATION
            # ------------------------------------------------
            save_visual(
                cover_path,
                stego_path,
                restored_rgb,
                payload_kbit,
                OUTPUT_PATH
            )

            # ------------------------------------------------
            # RESULT ROW
            # ------------------------------------------------
            row = build_result_row(
                image_name=image_name,
                payload_kbit=payload_kbit,
                metadata=meta,
                bits_original=orig_bits,
                bits_extracted=recv_bits,
                cover_rgb=cover_rgb,
                restored_rgb=restored_rgb
            )

            all_results.append(row)

            # ------------------------------------------------
            # LOG
            # ------------------------------------------------
            print(
                f'PSNR={row["PSNR"]:.2f} | '
                f'SSIM={row["SSIM"]:.6f} | '
                f'BER={row["BER"]:.10f} | '
                f'Embedded={row["Bits_Embedded"]:,}'
            )

        except Exception as e:

            print(
                f'[ERROR] '
                f'{image_name} | '
                f'{payload_kbit} Kbit'
            )

            print(str(e))

            continue

# ============================================================
# SAVE CSV
# ============================================================

csv_path = os.path.join(
    OUTPUT_PATH,
    'experiment_results.csv'
)

save_csv(
    all_results,
    csv_path
)

# ============================================================
# SUMMARY
# ============================================================

print('\n' + '=' * 80)
print('EXPERIMENT FINISHED')
print('=' * 80)

print(f'Total records : {len(all_results)}')
print(f'CSV saved     : {csv_path}')

MULTI IMAGE EVALUATION
Total images : 11

IMAGE : Abdomal.jpg

[Abdomal.jpg] Payload 1 Kbit
Payload loaded: random-binary_1Kb.txt (1000 bit)

[EMBEDDING] Abdomal.jpg
  Ukuran      : 512x512
  Payload     : 1,000 bit
  Target G/R/B: 330 / 320 / 350
[G] Embedded 330/330 bits  (100.00%) 
[R] Embedded 320/320 bits  (100.00%) 
[B] Embedded 350/350 bits  (100.00%) 
  Embedded G : 330
  Embedded R : 320
  Embedded B : 350
  Total      : 1,000/1,000
  Stego saved : D:\KBJ-Komputasi Jaringan\data hiding paper\propose fix\output_p1090\Abdomal_stego_1kbit.png
  PSNR = 71.3517 dB
  SSIM = 0.999997
  MSE  = 0.004763
  Metadata : D:\KBJ-Komputasi Jaringan\data hiding paper\propose fix\output_p1090\Abdomal_stego_1kbit_meta.json

[EXTRACTION + RECOVERY]
Image size : 512x512
Expected bits: G=330  R=320  B=350
[G] Extracted 330/330 bits
[R] Extracted 320/320 bits
[B] Extracted 350/350 bits

=== EXTRACTION REPORT ===
G : 330 bits
R : 320 bits
B : 350 bits
Total extracted : 1,000 bits
Recovered image : (5

In [25]:
# ============================================================
# GLOBAL REVERSIBILITY CHECK (FULL VERBOSE VERSION)
# ============================================================

def check_all_reversibility(results: list, show_all=False):

    print('\n' + '='*70)
    print('GLOBAL REVERSIBILITY CHECK (DETAILED)')
    print('='*70)

    if len(results) == 0:
        print('[ERROR] Tidak ada data untuk dicek')
        return False

    total = len(results)
    perfect = 0

    failed_cases = []

    # --------------------------------------------------------
    # LOOP SEMUA HASIL
    # --------------------------------------------------------
    for i, row in enumerate(results):

        img   = row.get('Image', 'Unknown')
        pld   = row.get('Payload_Kbit', 0)

        ber   = float(row.get('BER', 1))
        pxerr = int(row.get('Cover_Pixel_Error', 1))
        mdiff = int(row.get('Cover_Max_Diff', 1))

        msg_ok   = (ber == 0.0)
        cover_ok = (pxerr == 0 and mdiff == 0)
        is_ok    = msg_ok and cover_ok

        if is_ok:
            perfect += 1
        else:
            failed_cases.append({
                'Image': img,
                'Payload_Kbit': pld,
                'BER': ber,
                'Pixel_Error': pxerr,
                'Max_Diff': mdiff
            })

        # ----------------------------------------------------
        # PRINT PER CASE (OPTIONAL)
        # ----------------------------------------------------
        if show_all or (not is_ok):
            status = '✓' if is_ok else '✗'

            print(
                f'{status} {img:<15} | '
                f'{pld:>3} Kbit | '
                f'BER={ber:.2e} | '
                f'PxErr={pxerr:>5} | '
                f'MaxDiff={mdiff:>2}'
            )

    # --------------------------------------------------------
    # SUMMARY
    # --------------------------------------------------------
    print('\n' + '-'*70)
    print(f'Total cases        : {total}')
    print(f'Perfect reversible : {perfect}')
    print(f'Failed             : {total - perfect}')

    success_rate = 100 * perfect / total
    print(f'Success rate       : {success_rate:.2f}%')

    # --------------------------------------------------------
    # FINAL STATUS
    # --------------------------------------------------------
    if perfect == total:
        print('\n✓ ALL CASES PERFECT REVERSIBLE')
        return True

    # --------------------------------------------------------
    # DETAIL FAILED ONLY
    # --------------------------------------------------------
    print('\n[DETAIL FAILED CASES]')
    print('-'*70)

    for case in failed_cases:
        print(
            f"{case['Image']} | {case['Payload_Kbit']} Kbit | "
            f"BER={case['BER']:.2e} | "
            f"PxErr={case['Pixel_Error']} | "
            f"MaxDiff={case['Max_Diff']}"
        )

    return False

# ============================================================
# RUN GLOBAL CHECK
# ============================================================

if 'all_results' not in globals():
    print('[ERROR] all_results belum ada.')
elif len(all_results) == 0:
    print('[ERROR] all_results kosong.')
else:
    status = check_all_reversibility(
        all_results,
        show_all=True   # 🔥 TRUE = tampilkan semua case
    )

    print('\nFinal Status:', 'PASS' if status else 'FAIL')


GLOBAL REVERSIBILITY CHECK (DETAILED)
✓ Abdomal.jpg     |   1 Kbit | BER=0.00e+00 | PxErr=    0 | MaxDiff= 0
✓ Abdomal.jpg     |  10 Kbit | BER=0.00e+00 | PxErr=    0 | MaxDiff= 0
✓ Abdomal.jpg     |  20 Kbit | BER=0.00e+00 | PxErr=    0 | MaxDiff= 0
✓ Abdomal.jpg     |  30 Kbit | BER=0.00e+00 | PxErr=    0 | MaxDiff= 0
✓ Abdomal.jpg     |  40 Kbit | BER=0.00e+00 | PxErr=    0 | MaxDiff= 0
✓ Abdomal.jpg     |  50 Kbit | BER=0.00e+00 | PxErr=    0 | MaxDiff= 0
✓ Abdomal.jpg     |  60 Kbit | BER=0.00e+00 | PxErr=    0 | MaxDiff= 0
✓ Abdomal.jpg     |  70 Kbit | BER=0.00e+00 | PxErr=    0 | MaxDiff= 0
✓ Abdomal.jpg     |  80 Kbit | BER=0.00e+00 | PxErr=    0 | MaxDiff= 0
✓ Abdomal.jpg     |  90 Kbit | BER=0.00e+00 | PxErr=    0 | MaxDiff= 0
✓ Abdomal.jpg     | 100 Kbit | BER=0.00e+00 | PxErr=    0 | MaxDiff= 0
✓ Chest.jpg       |   1 Kbit | BER=0.00e+00 | PxErr=    0 | MaxDiff= 0
✓ Chest.jpg       |  10 Kbit | BER=0.00e+00 | PxErr=    0 | MaxDiff= 0
✓ Chest.jpg       |  20 Kbit | BER=0.0

In [26]:
# ============================================================
# CELL 13: SUMMARY + CSV REPORT
# ============================================================

print('\n' + '=' * 80)
print('FINAL EVALUATION SUMMARY')
print('=' * 80)

print(f'Total experiment results : {len(all_results)}')
print(f'Channel Weight           : {CHANNEL_WEIGHT}')
print(f'MAX_H                    : {MAX_H}')

# ------------------------------------------------------------
# NO RESULT
# ------------------------------------------------------------
if len(all_results) == 0:

    print('\n[WARN] Tidak ada hasil evaluasi.')

else:

    df = pd.DataFrame(all_results)

    # --------------------------------------------------------
    # REVERSIBLE STATUS
    # --------------------------------------------------------
    if (
        'BER' in df.columns and
        'Cover_Exact' in df.columns
    ):
        df['Reversible'] = (
            (df['BER'] == 0.0) &
            (df['Cover_Exact'] == True)
        )

    # --------------------------------------------------------
    # MAIN TABLE
    # --------------------------------------------------------
    print('\n=== MAIN RESULTS ===')

    main_cols = [
        c for c in [
            'Image',
            'Payload_Kbit',
            'PSNR',
            'SSIM',
            'MSE',
            'BER',
            'Cover_Exact',
            'Reversible'
        ]
        if c in df.columns
    ]

    print(
        df[main_cols]
        .round(6)
        .to_string(index=False)
    )

    # --------------------------------------------------------
    # EMBEDDING SUMMARY
    # --------------------------------------------------------
    if (
        'Bits_Target' in df.columns and
        'Bits_Embedded' in df.columns
    ):

        print('\n=== EMBEDDING UTILIZATION ===')

        emb_df = df[[
            'Image',
            'Payload_Kbit',
            'Bits_Target',
            'Bits_Embedded'
        ]].copy()

        emb_df['Utilization_%'] = np.where(
            emb_df['Bits_Target'] > 0,
            100 * emb_df['Bits_Embedded']
            / emb_df['Bits_Target'],
            0
        )

        print(
            emb_df
            .round(2)
            .to_string(index=False)
        )

    # --------------------------------------------------------
    # CHANNEL METRICS
    # --------------------------------------------------------
    channel_cols = [
        'Image',
        'Payload_Kbit',
        'PSNR_R',
        'PSNR_G',
        'PSNR_B'
    ]

    if all(c in df.columns for c in channel_cols):

        print('\n=== PSNR PER CHANNEL ===')

        print(
            df[channel_cols]
            .round(4)
            .to_string(index=False)
        )

    # --------------------------------------------------------
    # REVERSIBILITY
    # --------------------------------------------------------
    if 'Reversible' in df.columns:

        n_rev = int(df['Reversible'].sum())

        print('\n=== REVERSIBILITY ===')

        print(
            f'{n_rev}/{len(df)} '
            f'payload reversible'
        )

        if n_rev == len(df):
            print('✓ FULLY REVERSIBLE')
        else:
            print('✗ SOME PAYLOADS FAILED')

    # --------------------------------------------------------
    # GLOBAL STATISTICS
    # --------------------------------------------------------
    print('\n=== GLOBAL STATISTICS ===')

    if 'PSNR' in df.columns:
        print(
            f'PSNR Mean : {df["PSNR"].mean():.4f} dB'
        )
        print(
            f'PSNR Min  : {df["PSNR"].min():.4f} dB'
        )
        print(
            f'PSNR Max  : {df["PSNR"].max():.4f} dB'
        )

    if 'SSIM' in df.columns:
        print(
            f'SSIM Mean : {df["SSIM"].mean():.6f}'
        )

    if 'BER' in df.columns:
        print(
            f'BER Mean  : {df["BER"].mean():.10f}'
        )

    # --------------------------------------------------------
    # SAVE CSV
    # --------------------------------------------------------
    CSV_PATH = os.path.join(
        OUTPUT_PATH,
        'all_images_eval.csv'
    )

    try:

        df.to_csv(
            CSV_PATH,
            index=False
        )

        print('\n✓ CSV saved:')
        print(CSV_PATH)

    except Exception as e:

        print(
            f'\n[WARN] Failed saving CSV: {e}'
        )

print('\nDONE.')


FINAL EVALUATION SUMMARY
Total experiment results : 121
Channel Weight           : {'R': 0.32, 'G': 0.33, 'B': 0.35}
MAX_H                    : {'R': 10, 'G': 12, 'B': 12}

=== MAIN RESULTS ===
        Image  Payload_Kbit      PSNR     SSIM      MSE  BER  Cover_Exact  Reversible
  Abdomal.jpg             1 71.351738 0.999997 0.004763  0.0            1        True
  Abdomal.jpg            10 62.359997 0.999729 0.037764  0.0            1        True
  Abdomal.jpg            20 58.026059 0.999321 0.102441  0.0            1        True
  Abdomal.jpg            30 55.687051 0.998673 0.175540  0.0            1        True
  Abdomal.jpg            40 54.309169 0.998049 0.241081  0.0            1        True
  Abdomal.jpg            50 53.348902 0.997483 0.300739  0.0            1        True
  Abdomal.jpg            60 52.490084 0.996843 0.366498  0.0            1        True
  Abdomal.jpg            70 51.809106 0.996244 0.428716  0.0            1        True
  Abdomal.jpg            80 51.

In [27]:
# ============================================================
# CELL 14: CHARTS
# PSNR / SSIM / MSE vs Payload
# ============================================================

if len(all_results) == 0:

    print('[WARN] Tidak ada data untuk chart.')

else:

    df_plot = pd.DataFrame(all_results)

    required_cols = [
        'Image',
        'Payload_Kbit',
        'PSNR',
        'SSIM',
        'MSE'
    ]

    missing_cols = [
        c for c in required_cols
        if c not in df_plot.columns
    ]

    if len(missing_cols) > 0:

        print(
            f'[WARN] Missing columns: {missing_cols}'
        )

    else:

        print('\n=== GENERATING CHARTS ===')

        image_list = sorted(
            df_plot['Image'].unique()
        )

        for img_name in image_list:

            img_df = (
                df_plot[
                    df_plot['Image'] == img_name
                ]
                .sort_values('Payload_Kbit')
                .reset_index(drop=True)
            )

            payloads = img_df['Payload_Kbit'].tolist()

            psnr_vals = img_df['PSNR'].tolist()
            ssim_vals = img_df['SSIM'].tolist()
            mse_vals  = img_df['MSE'].tolist()

            x = np.arange(len(payloads))

            # ------------------------------------------------
            # FIGURE
            # ------------------------------------------------
            fig, axes = plt.subplots(
                1,
                3,
                figsize=(16, 5)
            )

            fig.suptitle(
                f'{img_name}\n'
                f'CHANNEL + DE Reversible + P10–P90',
                fontsize=13,
                fontweight='bold'
            )

            # =================================================
            # PSNR
            # =================================================
            axes[0].plot(
                payloads,
                psnr_vals,
                marker='o'
            )

            axes[0].axhline(
                30,
                linestyle='--'
            )

            axes[0].set_title('PSNR')
            axes[0].set_xlabel('Payload (Kbit)')
            axes[0].set_ylabel('PSNR (dB)')
            axes[0].grid(True)

            # =================================================
            # SSIM
            # =================================================
            axes[1].plot(
                payloads,
                ssim_vals,
                marker='o'
            )

            axes[1].axhline(
                0.99,
                linestyle='--'
            )

            axes[1].set_title('SSIM')
            axes[1].set_xlabel('Payload (Kbit)')
            axes[1].set_ylabel('SSIM')
            axes[1].grid(True)

            # =================================================
            # MSE
            # =================================================
            axes[2].plot(
                payloads,
                mse_vals,
                marker='o'
            )

            axes[2].set_title('MSE')
            axes[2].set_xlabel('Payload (Kbit)')
            axes[2].set_ylabel('MSE')
            axes[2].grid(True)

            plt.tight_layout()

            # ------------------------------------------------
            # SAVE
            # ------------------------------------------------
            img_base = os.path.splitext(
                img_name
            )[0]

            chart_path = os.path.join(
                OUTPUT_PATH,
                f'{img_base}_metrics.png'
            )

            try:

                plt.savefig(
                    chart_path,
                    dpi=300,
                    bbox_inches='tight'
                )

                print(
                    f'✓ Saved: '
                    f'{os.path.basename(chart_path)}'
                )

            except Exception as e:

                print(
                    f'[WARN] Save failed: {e}'
                )

            plt.close(fig)

        print('\n✓ All charts generated.')


=== GENERATING CHARTS ===
✓ Saved: Abdomal_metrics.png
✓ Saved: Chest_metrics.png
✓ Saved: Hand_metrics.png
✓ Saved: Head_metrics.png
✓ Saved: Leg_metrics.png
✓ Saved: airplane_metrics.png
✓ Saved: baboon_metrics.png
✓ Saved: house_metrics.png
✓ Saved: pepper_metrics.png
✓ Saved: sailboat_metrics.png
✓ Saved: splash_metrics.png

✓ All charts generated.
